In [1]:
# #set programmatically!
# video_dir = "/Users/cybellesmith/Box Sync/all/post_doc2/code/kiera_data_analysis/processed_videos"
# event_dir = "/Users/cybellesmith/Box Sync/all/post_doc2/code/kiera_data_analysis/event_detection_output"
# video_filename = "DIV27_HS1-DA1M_W9_optostim_mc_dff_resid.tif"

In [ ]:
import sys,os
import imagej
import scyjava as sj
import tifffile as tff
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import cv2
import re
import pandas as pd

In [ ]:
import re

match = re.search(r'W(\d+)', video_filename)
if match:
    well = int(match.group(1))
else:
    well = 0

In [ ]:
fps = 1.849

if well == 7:
    fps = 0.939

In [ ]:
video_full_filename = re.sub(r"\_dff_resid.tif$", ".tif", video_filename)
print(video_full_filename)
mask_filename = re.sub(r"\.tif$", "_global_mask.tif", video_filename)
print(mask_filename)
output_csv_filename = re.sub(r"\_mc_dff_resid.tif$", "_caiman_events.csv", video_filename) 
print(output_csv_filename)

In [ ]:
#step 1: precrop the video, fill pixels outside active ROI with noise...
#must have a rectangle that is totally filled with the ROI and pixels that are active with "noise" since
#CaImAn pipeline designed for miniscopes not in vitro work

os.chdir(video_dir)

video = tff.imread(video_filename)

mask = tff.imread(mask_filename)

plt.imshow(video[0], cmap="gray")
plt.show()
plt.imshow(mask, cmap="gray")
plt.show()


In [ ]:
#crop mask and video

import numpy as np

def bounding_box_2d(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None  # no True pixels
    
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    
    return (y_min, y_max, x_min, x_max)

bbox = bounding_box_2d(mask)

y0, y1, x0, x1 = bbox

cropped_mask = mask[y0:y1+1, x0:x1+1]
cropped_video = np.array([video[t, y0:y1+1, x0:x1+1] for t in range(len(video))])

plt.imshow(cropped_video[0], cmap="gray")
plt.show()
plt.imshow(cropped_mask, cmap="gray")
plt.show()

In [ ]:
pix_std = np.std(cropped_video, axis=0)

plt.imshow(pix_std)
plt.show()

In [ ]:
bad = pix_std == 0
print(np.sum(bad))

noise_level = np.median(pix_std[pix_std > 0])

rng = np.random.default_rng(0)

for t in range(video.shape[0]):
    cropped_video[t, bad] = rng.normal(0, noise_level, np.sum(bad))

plt.imshow(cropped_video[0], cmap="gray")
plt.show()

In [ ]:
#make nonnegative
cropped_video = cropped_video + np.min(cropped_video)

In [ ]:
import re
out_file = re.sub(r'\.tif$', '_cropped_noise_added_nonneg.tif', video_filename)
tff.imwrite(out_file, cropped_video)

In [ ]:
import bokeh.plotting as bpl
import cv2
import glob
import holoviews as hv
from IPython import get_ipython
import logging
import matplotlib.pyplot as plt
import numpy as np
import os
import psutil

import caiman as cm
from caiman.source_extraction import cnmf
from caiman.source_extraction.cnmf.cnmf import load_CNMF
from caiman.utils.utils import download_demo
from caiman.utils.visualization import inspect_correlation_pnr, nb_inspect_correlation_pnr
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import params as params
from caiman.utils.visualization import plot_contours, nb_view_patches, nb_plot_contour
from caiman.utils.visualization import view_quilt

try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
        # get_ipython().run_line_magic('matplotlib', 'qt')  #uncomment to run in qt mode
except NameError:
    pass

try:
    cv2.setNumThreads(0)
except:
    pass

bpl.output_notebook()
hv.notebook_extension('bokeh')

# Set play_movies to False if you want to disable play of movies, e.g. for remote-hosted Jupyter environments
play_movies = True

In [ ]:
# set up logging
logfile = None # Replace with a path if you want to log to a file
logger = logging.getLogger('caiman')
# Set to logging.INFO if you want much output, potentially much more output
logger.setLevel(logging.WARNING)
logfmt = logging.Formatter('%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s')
if logfile is not None:
    handler = logging.FileHandler(logfile)
else:
    handler = logging.StreamHandler()
handler.setFormatter(logfmt)
logger.addHandler(handler)

# set env variables in case they weren't already set
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

In [ ]:
movie_path = video_dir + "/" + out_file

In [ ]:
# # press q to close
# movie_orig = cm.load(movie_path) 
# downsampling_ratio = 0.2  # subsample 5x
# movie_orig.resize(fz=downsampling_ratio)
# if play_movies:
#     movie_orig.play(gain=0.9,
#                     q_max=99.5, 
#                     fr=10,
#                     plot_text=True,
#                     magnification=4,
#                     do_loop=True,
#                     backend='opencv')

In [ ]:
print(f"You have {psutil.cpu_count()} CPUs available in your current environment")
num_processors_to_use = 8

In [ ]:

#%% start a cluster for parallel processing (if a cluster already exists it will be closed and a new session will be opened)
if 'cluster' in locals():  # 'locals' contains list of current local variables
    print('Closing previous cluster')
    cm.stop_server(dview=cluster)
print("Setting up new cluster")
_, cluster, n_processes = cm.cluster.setup_cluster(backend='multiprocessing', 
                                                 n_processes=num_processors_to_use, 
                                                 ignore_preexisting=False)
print(f"Successfully set up cluster with {n_processes} processes")

In [ ]:
dview = cluster

In [ ]:
fname_new = cm.save_memmap([movie_path], base_name='memmap_',
                           order='C', border_to_0=0, dview=dview)

In [ ]:
print(movie_path)
print(fname_new)

In [ ]:
# load memory mappable file
Yr, dims, T = cm.load_memmap(fname_new)
images = Yr.T.reshape((T,) + dims, order='F')

In [ ]:
gsig_tmp = (2,2)
correlation_image, peak_to_noise_ratio = cm.summary_images.correlation_pnr(images[::max(T//1000, 1)], # subsample if needed
                                                                           gSig=gsig_tmp[0], # used for filter
                                                                           swap_dim=False) # change swap dim if output looks weird, it is a problem with tiffile

In [ ]:
#I think this might break something later?

# import bokeh.plotting as bpl
# import holoviews as hv
# from IPython.display import display

# bpl.output_notebook()
# hv.notebook_extension('bokeh')

# obj = nb_inspect_correlation_pnr(correlation_image, peak_to_noise_ratio, cmap='inferno')
# display(obj)

In [ ]:
inspect_correlation_pnr(correlation_image, peak_to_noise_ratio)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.imshow(correlation_image, cmap='inferno')
plt.show()
plt.hist(correlation_image.flatten(), bins=200)
plt.show()

plt.imshow(peak_to_noise_ratio, cmap='inferno')
plt.show()

pnr_flat = peak_to_noise_ratio.flatten()
plt.hist(pnr_flat, bins=200)
plt.show()
plt.hist(pnr_flat[pnr_flat < 100], bins=200)
plt.show()
plt.hist(pnr_flat[pnr_flat < 20], bins=200)
plt.show()

In [ ]:
# parameters for source extraction and deconvolution
p = 1               # order of the autoregressive system
K = None            # upper bound on number of components per patch, in general None for CNMFE
gSig = np.array([3, 3])  # expected half-width of neurons in pixels 
gSiz = 2*gSig + 1     # half-width of bounding box created around neurons during initialization
merge_thr = .7      # merging threshold, max correlation allowed
rf = 81             # half-size of the patches in pixels. e.g., if rf=40, patches are 80x80
stride_cnmf = 60    # amount of overlap between the patches in pixels 
tsub = 1            # downsampling factor in time for initialization, increase if you have memory problems
ssub = 1            # downsampling factor in space for initialization, increase if you have memory problems
gnb = 0             # number of background components (rank) if positive, set to 0 for CNMFE
low_rank_background = None  # None leaves background of each patch intact (use True if gnb>0)
nb_patch = 0        # number of background components (rank) per patch (0 for CNMFE)
min_corr = .4       # min peak value from correlation image
min_pnr = 7        # min peak to noise ratio from PNR image
ssub_B = 1          # additional downsampling factor in space for background (increase to 2 if slow)
ring_size_factor = 1.2  # radius of ring is gSiz*ring_size_factor
bord_px = 0
init_iter = 1
fr = fps #frame rate (frames per second)
decay_time = 1 #duration of a typical calcium transient in seconds

parameters = params.CNMFParams(params_dict={'method_init': 'corr_pnr',  # use this for 1 photon
                                'K': K,
                                'gSig': gSig,
                                'gSiz': gSiz,
                                'merge_thr': merge_thr,
                                'p': p,
                                'tsub': tsub,
                                'ssub': ssub,
                                'rf': rf,
                                'stride': stride_cnmf,
                                'only_init': True,    # set it to True to run CNMF-E
                                'nb': gnb,
                                'nb_patch': nb_patch,
                                'method_deconvolution': 'oasis',       # could use 'cvxpy' alternatively
                                'low_rank_background': low_rank_background,
                                'update_background_components': True,  # sometimes setting to False improve the results
                                'min_corr': min_corr,
                                'min_pnr': min_pnr,
                                'normalize_init': False,               # just leave as is
                                'center_psf': True,                    # True for 1p
                                'ssub_B': ssub_B,
                                'ring_size_factor': ring_size_factor,
                                'del_duplicates': True,                # whether to remove duplicates from initialization
                                'border_pix': bord_px,
                                'init_iter': init_iter,
                                'fr': fr, #frame rate (frames per second)
                                'decay_time': decay_time #duration of a typical calcium transient in seconds
                                });                # number of pixels to not consider in the borders)


In [ ]:
cnmfe_model = cnmf.CNMF(n_processes=n_processes, 
                        dview=cluster, 
                        params=parameters)

In [ ]:
# estimate stride and overlap from parameters
cnmfe_patch_width = cnmfe_model.params.patch['rf']*2 + 1
cnmfe_patch_overlap = cnmfe_model.params.patch['stride'] + 1
cnmfe_patch_stride = cnmfe_patch_width - cnmfe_patch_overlap
print(f'Patch width: {cnmfe_patch_width} , Stride: {cnmfe_patch_stride}, Overlap: {cnmfe_patch_overlap}');

# plot the patches
patch_ax = view_quilt(correlation_image, 
                      cnmfe_patch_stride, 
                      cnmfe_patch_overlap, 
                      vmin=np.percentile(np.ravel(correlation_image), 50), 
                      vmax=np.percentile(np.ravel(correlation_image), 99.5),
                      color='yellow',
                      figsize=(4,4));
patch_ax.set_title(f'CNMFE Patch Width {cnmfe_patch_width}, Overlap {cnmfe_patch_overlap}');


In [ ]:
%%time
cnmfe_model.fit(images);

In [ ]:

#try this but in the end you will not use -- you will include all detected components!

min_SNR = 3           # SNR threshold
rval_thr = 0.5    # spatial correlation threshold

quality_params = {
    'quality': {
        'min_SNR': min_SNR,
        'rval_thr': rval_thr,
        'use_cnn': False
    }
}
cnmfe_model.params.change_params(params_dict=quality_params)

if cnmfe_model.estimates.C is None or cnmfe_model.estimates.YrA is None:
    raise RuntimeError(
        "CNMF-E fit did not produce temporal traces/residuals. "
        "Check earlier fit steps before calling evaluate_components()."
    )
    
cnmfe_model.estimates.evaluate_components(images, cnmfe_model.params, dview=cluster)

print('*****')
print(f"Total number of components: {len(cnmfe_model.estimates.C)}")
print(f"Number accepted: {len(cnmfe_model.estimates.idx_components)}")
print(f"Number rejected: {len(cnmfe_model.estimates.idx_components_bad)}")

In [ ]:
#%% plot contour plots of accepted and rejected components
cnmfe_model.estimates.plot_contours(img=correlation_image, 
                                       idx=cnmfe_model.estimates.idx_components, display_numbers=False);

In [ ]:
#doesn't always work??

# # accepted components
# cnmfe_model.estimates.nb_view_components(img=correlation_image, 
#                                         idx=cnmfe_model.estimates.idx_components,
#                                         cmap='viridis', #gray
#                                         denoised_color='red',
#                                         thr=.9 #increase to see full footprint
#                                         ); 

In [ ]:
# # rejected components
# cnmfe_model.estimates.nb_view_components(img=correlation_image, 
#                                         idx=cnmfe_model.estimates.idx_components_bad,
#                                         cmap='viridis', #gray
#                                         denoised_color='red',
#                                         thr=0.9); #increase to see full footprint

In [ ]:
save_results = True
if save_results:
    save_path =  rf'well{well}_resid_cnmfe_results.hdf5'  # or add full/path/to/file.hdf5
    cnmfe_model.estimates.Cn = correlation_image # squirrel away correlation image with cnmf object
    cnmfe_model.save(save_path)

In [ ]:
load_results = True
if load_results:
    save_path =  rf'well{well}_resid_cnmfe_results.hdf5'  # or add full/path/to/file.hdf5
    cnmfe_model = load_CNMF(save_path, 
                                n_processes=num_processors_to_use, 
                                dview=cluster)
    correlation_image = cnmfe_model.estimates.Cn
    print(f"Successfully loaded data.")

In [ ]:
# in case you are working from loaded data, recover the raw movie
Yr, dims, num_frames = cm.load_memmap(cnmfe_model.mmap_file)
images = np.reshape(Yr.T, [num_frames] + list(dims), order='F')

In [ ]:
# neural_activity = cnmfe_model.estimates.A[:, cnmfe_model.estimates.idx_components] @\
#                   cnmfe_model.estimates.C[cnmfe_model.estimates.idx_components, :]  # AC

In [ ]:
neural_activity = cnmfe_model.estimates.A[:, :] @\
                  cnmfe_model.estimates.C[:, :]  # AC
neural_movie = cm.movie(neural_activity).reshape(dims + (-1,), order='F').transpose([2, 0, 1])
background_model = cnmfe_model.estimates.compute_background(Yr);  # build in function -- explore source code for details
bg_movie = cm.movie(background_model).reshape(dims + (-1,), order='F').transpose([2, 0, 1])

In [ ]:
# downsampling_ratio = 0.4 
# neural_movie.resize(fz=downsampling_ratio)
# if play_movies:
#     neural_movie.play(gain=1.1,
#                       q_max=99.5, 
#                       fr=20,
#                       plot_text=True,
#                       magnification=2,
#                       do_loop=False,
#                       backend='opencv')

In [ ]:
# downsampling_ratio = 0.8 
# bg_movie.resize(fz=downsampling_ratio)
# if play_movies:
#     bg_movie.play(gain=1.1,
#                   q_max=99.5,
#                   fr=10,
#                   plot_text=True,
#                   magnification=2,
#                   do_loop=False,
#                   backend='opencv')

In [ ]:
# # without background
# cnmfe_model.estimates.play_movie(
#     images,
#     frame_range=slice(0, 200),
#     magnification=2,
#     include_bck=True,
#     gain_res=1,
#     use_color=False
# )

In [ ]:
os.chdir(video_dir)

video_full = tff.imread(video_full_filename)

plt.imshow(video_full[0], cmap="gray")
plt.show()
plt.imshow(mask, cmap="gray")
plt.show()

In [ ]:
#crop mask and video

import numpy as np

def bounding_box_2d(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None  # no True pixels
    
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    
    return (y_min, y_max, x_min, x_max)

bbox = bounding_box_2d(mask)

y0, y1, x0, x1 = bbox

cropped_mask = mask[y0:y1+1, x0:x1+1]
cropped_video_full = np.array([video_full[t, y0:y1+1, x0:x1+1] for t in range(len(video_full))])

plt.imshow(cropped_video_full[0], cmap="gray")
plt.show()
plt.imshow(cropped_mask, cmap="gray")
plt.show()

In [ ]:
pix_std = np.std(cropped_video_full, axis=0)

bad = pix_std == 0

np.sum(bad)

In [ ]:
neural_activity_all_comp = cnmfe_model.estimates.A[:, :] @\
                  cnmfe_model.estimates.C[:, :]  # AC

In [ ]:
(T, H, W) = cropped_video.shape
out = neural_activity.reshape(W, H , T).transpose(2, 1, 0)
for t in range(len(out)):
    out[t][bad] = 0
    
os.chdir(video_dir)
out_file = re.sub(r'\.tif$', '_caiman_neural_activity_inferred.tif', video_filename)
tff.imwrite(out_file, out)

In [ ]:
#ok - try to merge propagation events across components
#step 1 build ROI graph (spatially adjacent candidate "cells")

import numpy as np
from scipy import sparse
from scipy import ndimage as ndi


def caiman_A_to_masks(A, dims, thr_frac=0.2):
    """
    Convert CaImAn spatial components A into binary ROI masks.

    Parameters
    ----------
    A : np.ndarray or scipy sparse matrix, shape (H*W, K)
        CaImAn spatial footprints.
    dims : tuple
        (H, W)
    thr_frac : float
        Threshold each component at thr_frac * max(component).

    Returns
    -------
    masks : np.ndarray, shape (K, H, W), dtype=bool
    """
    H, W = dims

    if sparse.issparse(A):
        A = A.tocsc()

    K = A.shape[1]
    masks = np.zeros((K, H, W), dtype=bool)

    for k in range(K):
        ak = A[:, k].toarray().reshape(H, W, order="F") if sparse.issparse(A) else A[:, k].reshape(H, W, order="F")
        mx = ak.max()
        if mx <= 0:
            continue
        masks[k] = ak >= (thr_frac * mx)

    return masks


def compute_centroids_from_masks(masks):
    """
    Centroid of each binary mask as (y, x).
    """
    K = masks.shape[0]
    centroids = np.full((K, 2), np.nan, dtype=float)

    for k in range(K):
        yy, xx = np.nonzero(masks[k])
        if len(yy) == 0:
            continue
        centroids[k] = [yy.mean(), xx.mean()]

    return centroids


def min_mask_distance(mask1, mask2):
    """
    Minimum Euclidean distance (in pixels) between two binary masks.
    Returns 0 if they overlap.
    """
    y1, x1 = np.nonzero(mask1)
    y2, x2 = np.nonzero(mask2)

    if len(y1) == 0 or len(y2) == 0:
        return np.inf

    pts1 = np.column_stack([y1, x1])
    pts2 = np.column_stack([y2, x2])

    d2 = ((pts1[:, None, :] - pts2[None, :, :]) ** 2).sum(axis=2)
    return np.sqrt(d2.min())


def build_roi_graph(
    A,
    dims,
    thr_frac=0.2,
    dilation_iter=1,
    max_dist=3.0,
    use_centroid_gate=True,
    centroid_gate_dist=25.0,
):
    """
    Build an ROI adjacency graph from CaImAn spatial footprints.

    Two ROIs are connected if:
      - their masks overlap after dilation, OR
      - min mask distance <= max_dist

    Optionally uses centroid distance as a cheap pre-filter.

    Parameters
    ----------
    A : np.ndarray or scipy sparse matrix, shape (H*W, K)
        CaImAn spatial footprints.
    dims : tuple
        (H, W)
    thr_frac : float
        Threshold fraction for binarizing each ROI.
    dilation_iter : int
        Number of binary dilation iterations before overlap test.
    max_dist : float
        Maximum allowed minimum pixel distance to connect ROIs.
    use_centroid_gate : bool
        If True, skip expensive mask-distance checks for pairs whose
        centroids are farther than centroid_gate_dist.
    centroid_gate_dist : float
        Distance threshold for centroid pre-filter.

    Returns
    -------
    roi_graph : np.ndarray, shape (K, K), dtype=bool
        Symmetric adjacency matrix.
    centroids : np.ndarray, shape (K, 2)
        ROI centroids as (y, x).
    masks : np.ndarray, shape (K, H, W), dtype=bool
        Binary ROI masks.
    """
    masks = caiman_A_to_masks(A, dims=dims, thr_frac=thr_frac)
    centroids = compute_centroids_from_masks(masks)

    K = masks.shape[0]
    roi_graph = np.zeros((K, K), dtype=bool)

    # pre-dilate for overlap test
    if dilation_iter > 0:
        dil_masks = np.array([
            ndi.binary_dilation(m, iterations=dilation_iter) for m in masks
        ])
    else:
        dil_masks = masks.copy()

    for i in range(K):
        for j in range(i + 1, K):
            if np.any(np.isnan(centroids[i])) or np.any(np.isnan(centroids[j])):
                continue

            if use_centroid_gate:
                dc = np.linalg.norm(centroids[i] - centroids[j])
                if dc > centroid_gate_dist:
                    continue

            # overlap after dilation?
            if np.any(dil_masks[i] & dil_masks[j]):
                roi_graph[i, j] = True
                roi_graph[j, i] = True
                continue

            # otherwise check min mask distance
            dmin = min_mask_distance(masks[i], masks[j])
            if dmin <= max_dist:
                roi_graph[i, j] = True
                roi_graph[j, i] = True

    return roi_graph, centroids, masks

In [ ]:
roi_graph, centroids, masks = build_roi_graph(
    cnmfe_model.estimates.A,
    cropped_video[0].shape,
    thr_frac=0.2,
    dilation_iter=1,
    max_dist=3.0,
    use_centroid_gate=True,
    centroid_gate_dist=25.0,
)

In [ ]:
plt.imshow(roi_graph)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_roi_graph(roi_graph, centroids, background=None):
    plt.figure(figsize=(8, 8))
    if background is not None:
        plt.imshow(background, cmap="gray")

    for i in range(len(centroids)):
        y, x = centroids[i]
        if np.isnan(y):
            continue
        plt.plot(x, y, 'ro', ms=3)
        #plt.text(x + 1, y + 1, str(i), color='yellow', fontsize=8)

    ii, jj = np.nonzero(np.triu(roi_graph, 1))
    for i, j in zip(ii, jj):
        y1, x1 = centroids[i]
        y2, x2 = centroids[j]
        plt.plot([x1, x2], [y1, y2], 'c-', lw=0.7)

    plt.gca().invert_yaxis()
    plt.axis("equal")
    plt.tight_layout()
    plt.show()

plot_roi_graph(roi_graph, centroids, correlation_image)

In [ ]:
# define helper function for detrending

import numpy as np
from scipy import sparse
from scipy.sparse.linalg import spsolve

def asls_baseline(y, lam=1e5, p=0.01, niter=10):
    """
    y: 1D array
    lam: smoothness (↑ -> smoother baseline)
    p: asymmetry (small -> treat positive peaks as outliers)
    """
    y = np.asarray(y, float)
    L = y.size
    D = sparse.diags([1, -2, 1], [0, 1, 2], shape=(L-2, L), format="csc")
    w = np.ones(L)
    for _ in range(niter):
        W = sparse.diags(w, 0, shape=(L, L))
        Z = W + lam * (D.T @ D)
        z = spsolve(Z, w*y)
        w = p * (y > z) + (1-p) * (y <= z)  # downweight positive residuals
    return z

In [ ]:
def robust_std(signal, eps=1e-6):
    med = np.median(signal)                           # (Y,X)
    mad = np.median(np.abs(signal - med))             # 
    sigma = 1.4826 * mad                                     # robust std estimate
    return sigma

In [ ]:
eps = 0.1
all_detr_norm = []
for F in cnmfe_model.estimates.C:
    base = asls_baseline(F, lam=1e5, p=0.01, niter=10) #lam=1e5
    detrended = F - base
    if np.sum((detrended > eps) | (detrended < -eps)) > 0:
        detr_norm = detrended / robust_std(detrended[(detrended > eps) | (detrended < -eps)])
    else:
        detr_norm = detrended / robust_std(detrended)
    all_detr_norm.append(detr_norm)

In [ ]:
from scipy.signal import savgol_filter

wlen = 51
if well == 7:
    wlen = 25 #closest odd integer accounting for frame rate diff
all_smoothed = []
for signal in all_detr_norm:
    all_smoothed.append(savgol_filter(signal, window_length=wlen, polyorder=3))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import spectrogram

def fill_nans_1d(x):
    x = np.asarray(x, dtype=float).copy()
    n = len(x)

    bad = ~np.isfinite(x)
    if bad.all():
        return np.zeros_like(x)

    if bad.any():
        good_idx = np.flatnonzero(~bad)
        bad_idx = np.flatnonzero(bad)
        x[bad_idx] = np.interp(bad_idx, good_idx, x[good_idx])

    return x

def mean_spectrogram(signals, fs, nperseg=128, noverlap=64, average_log=False):
    signals = np.asarray(signals, dtype=float)

    if signals.ndim != 2:
        raise ValueError(f"signals must have shape (n_signals, T), got {signals.shape}")

    n_signals, T = signals.shape

    if T < 2:
        raise ValueError("Signals are too short.")

    nperseg = min(nperseg, T)
    noverlap = min(noverlap, nperseg - 1)

    specs = []
    kept_rows = []

    for i, s in enumerate(signals):
        if np.all(~np.isfinite(s)):
            continue

        s = fill_nans_1d(s)

        f, t, S = spectrogram(
            s,
            fs=fps,
            nperseg=nperseg,
            noverlap=noverlap,
            scaling="density",
            mode="psd",
        )

        if np.all(~np.isfinite(S)):
            print(f"row {i}: spectrogram is all non-finite")
            continue

        specs.append(S)
        kept_rows.append(i)

    if len(specs) == 0:
        raise ValueError("No valid spectrograms were computed. Check for NaNs or invalid traces.")

    specs = np.stack(specs, axis=0)

    if average_log:
        S_mean = np.exp(np.nanmean(np.log(specs + 1e-12), axis=0))
    else:
        S_mean = np.nanmean(specs, axis=0)

    return f, t, S_mean, kept_rows

def plot_mean_spectrogram(f, t, S):
    plt.figure(figsize=(6, 4))
    plt.pcolormesh(t, f, 10 * np.log10(S + 1e-12), shading="auto")
    plt.xlabel("Time")
    plt.ylabel("Frequency")
    plt.title("Mean Spectrogram")
    plt.colorbar(label="Power (dB)")
    plt.tight_layout()
    plt.show()

In [ ]:
signals = all_smoothed.copy()
if np.sum(np.isnan(signals)) > 0:
    signals = np.array([fill_nans_1d(signal) for signal in signals])

In [ ]:
signals_norm = [s / (np.nanmax(s) - np.nanmin(s)) for s in signals]

In [ ]:

f, t, S, kept_rows = mean_spectrogram(
    signals_norm,
    fs=fps,
    nperseg=128,
    noverlap=32,
    average_log=False,
)

plot_mean_spectrogram(f, t, S)

In [ ]:
f

In [ ]:
mean_signal = np.nanmean(signals_norm,axis=0)
plt.plot(mean_signal)
plt.show()

In [ ]:
signal_intercepts = []
entry_pts = []
exit_pts = []

T = len(signals[0])

eps = 0.0

for i in range(len(signals)):
    signal = signals[i]
    #eps = thresholds[i]
    cross_bool = np.diff(np.sign(signal - eps))
    tmp = [[0], np.where(cross_bool)[0], [T-1]]
    signal_intercept = [j for i in tmp for j in i] #add 0 and final index
    signal_intercept = list(set(signal_intercept)) #remove duplicate values (in case 0 or final index already are zero crossings)
    signal_intercept = list(np.sort(np.array(signal_intercept))) #sort intercepts from low to high
    signal_intercepts.append(signal_intercept)
    entry_pts.append(np.where(cross_bool > 0)[0])
    exit_pts.append(np.where(cross_bool < 0)[0])

In [ ]:
A = cnmfe_model.estimates.A
dims = cnmfe_model.dims  # (height, width)

spatial_maps = []

for i in range(len(cnmfe_model.estimates.C)):
    spatial_map = A[:, i].toarray().reshape(dims, order='F')
    spatial_maps.append(spatial_map)

In [ ]:
#plot signal intercepts:
n_roi = len(signals)
fig, axs = plt.subplots(n_roi,2)
fig.suptitle('Signal Initial Segmentation\n(Boundaries > Min, First & Last Sample)')
fig.set_size_inches(6, n_roi * 2)

T = len(signals[0])

time_x = [i for i in range(T)]

eps = .1
thresh = 0.0

#plot signal columns:
for i in range(len(signals)):
    s = i
    ymin = np.round(np.nanmin(np.array(signals[s])) - eps,1)
    ymax = np.round(np.nanmax(np.array(signals[s])) + eps,1)

    axs[i,0].plot(time_x,signals[s], color="black")
    for intercept in signal_intercepts[s]:
        axs[i,0].axvline(intercept, color="blue")
    axs[i,0].axhline(y=thresh, color='red', linestyle='dotted')
    axs[i,0].set_ylim(ymin,ymax)
    chimg = spatial_maps[i]
    axs[i,1].imshow(chimg)

plt.subplots_adjust(top=0.96)
plt.show()

In [ ]:
#if subsignal > threshold within boundary set by threshold, count as an event!

n_roi = len(signals)

event_thresh = 0.5
#event_thresh = 1.0

init_events = []
for s in range(n_roi):
    roi_init_events = []
    #event_thresh = thresholds[s] + extra_thresh
    if len(signal_intercepts[s]) > 0:
        for i, intercept in enumerate(signal_intercepts[s][0:(len(signal_intercepts[s]) - 1)]):
            start_idx = intercept
            end_idx = signal_intercepts[s][i + 1]
            subsignal = signals[s][start_idx:end_idx]
            if np.max(subsignal) > event_thresh:
                roi_init_events.append((start_idx,end_idx))

    init_events.append(roi_init_events)
init_events

In [ ]:
#plot detected events

n_roi = len(signals)
fig, axs = plt.subplots(n_roi,2)
fig.suptitle('Detected Events')
fig.set_size_inches(6, n_roi * 2)

T = len(signals[0])

time_x = [i for i in range(T)]

eps = .1

#plot signal columns:
for i in range(len(signals)):
    s = i
    ymin = np.round(np.nanmin(np.array(signals[s])) - eps,1)
    ymax = np.round(np.nanmax(np.array(signals[s])) + eps,1)

    axs[i,0].plot(time_x,signals[s], color="black")
    for event in init_events[s]:
        axs[i,0].axvspan(event[0], event[1], color='yellow', alpha=0.3)
    axs[i,0].axhline(y=event_thresh, color='red', linestyle='dotted')
    axs[i,0].set_ylim(ymin,ymax)
    chimg = spatial_maps[i]
    axs[i,1].imshow(chimg)

plt.subplots_adjust(top=0.96)
plt.show()

In [ ]:
events = []
for s in range(len(signals)):
    for init_event in init_events[s]:
        subsignal = signals[s][init_event[0]:init_event[1]]
        tp = init_event[0] + np.argmax(subsignal)
        amp = np.max(subsignal)
        event = {
            
                    "roi": s,
                    "t0": init_event[0],
                    "tp": tp,
                    "t1": init_event[1],
                    "amp": amp,
                    "t0_sec": init_event[0] / fps,
                    "tp_sec": tp / fps,
                    "t1_sec": init_event[1] / fps,
                    "duration_sec": (init_event[1] - init_event[0]) / fps
            
                }
        events.append(event)

In [ ]:
events

In [ ]:
def intervals_close(t0_a, t1_a, t0_b, t1_b, max_gap=0):
    """
    Return True if two intervals overlap or are within max_gap frames.

    Parameters
    ----------
    t0_a, t1_a : start/end of interval A
    t0_b, t1_b : start/end of interval B
    max_gap : allowed gap between intervals
    """

    if t0_a > t1_a:
        t0_a, t1_a = t1_a, t0_a
    if t0_b > t1_b:
        t0_b, t1_b = t1_b, t0_b

    gap = max(t0_a, t0_b) - min(t1_a, t1_b)

    return gap <= max_gap

In [ ]:
# one parent per event
parent = list(range(len(events)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]  # path compression
        x = parent[x]
    return x

def union(a, b):
    ra = find(a)
    rb = find(b)
    if ra != rb:
        parent[rb] = ra

In [ ]:
max_peak_lag = 5
max_gap = 5

if well == 7:
    max_peak_lag = 3
    max_gap = 3

# event graph
n_events = len(events)

for i in range(n_events):
    ei = events[i]
    roi_i = ei["roi"]

    for j in range(i + 1, n_events):
        ej = events[j]
        roi_j = ej["roi"]

        # only compare events from spatially adjacent ROIs
        if not roi_graph[roi_i, roi_j]:
            continue

        # peaks must be close in time
        if abs(ei["tp"] - ej["tp"]) > max_peak_lag:
            continue

        # event windows must overlap or nearly touch
        if not intervals_close(ei["t0"], ei["t1"], ej["t0"], ej["t1"], max_gap):
            continue

        union(i, j)

In [ ]:
groups = {}
for i in range(n_events):
    root = find(i)
    groups.setdefault(root, []).append(i)

merged_event_groups = list(groups.values())

In [ ]:
len(events)

In [ ]:
len(merged_event_groups)

In [ ]:
merged_events = [[events[i] for i in grp] for grp in merged_event_groups]

In [ ]:
merged_events

In [ ]:
merged_events_filtered = [me for me in merged_events if len(me) > 1]

In [ ]:
merged_events_filtered

In [ ]:
#faster version of cell above:

import numpy as np
from scipy import sparse

T, H, W = cropped_video.shape
dims = (H, W)

neural_events_singleton = np.zeros((T, H, W), dtype=np.float32)
neural_events_merged    = np.zeros((T, H, W), dtype=np.float32)
neural_events_all       = np.zeros((T, H, W), dtype=np.float32)

A = cnmfe_model.estimates.A
C = np.asarray(cnmfe_model.estimates.C, dtype=np.float32)

K = C.shape[0]

# Precompute all ROI footprints once
roi_imgs = np.zeros((K, H, W), dtype=np.float32)

for roi in range(K):
    a = A[:, roi]
    if sparse.issparse(a):
        a = a.toarray().ravel()
    else:
        a = np.asarray(a).ravel()

    roi_imgs[roi] = a.reshape((H, W), order="F")

for event in merged_events:
    target = neural_events_merged if len(event) > 1 else neural_events_singleton

    for linked_event in event:
        roi = linked_event["roi"]
        start = linked_event["t0"]
        stop = linked_event["t1"]   # use +1 here if t1 is meant to be inclusive

        if stop <= start:
            continue

        spatial = roi_imgs[roi]               # (H, W)
        temporal = C[roi, start:stop]         # (L,)

        # (L, 1, 1) * (H, W) -> (L, H, W)
        activity = temporal[:, None, None] * spatial[None, :, :]

        neural_events_all[start:stop] += activity
        target[start:stop] += activity

In [ ]:
out_file = re.sub(r'\.tif$', '_caiman_all_events.tif', video_filename)
tff.imwrite(out_file, neural_events_all)

In [ ]:
import numpy as np
import tifffile as tiff

# assume:
# neural_events_singleton.shape == (T, H, W)
# neural_events_merged.shape    == (T, H, W)

singleton = np.asarray(neural_events_singleton, dtype=np.float32)
merged = np.asarray(neural_events_merged, dtype=np.float32)

# optional: clip negatives if present
singleton = np.maximum(singleton, 0)
merged = np.maximum(merged, 0)

# scale each channel separately to 0..255
def scale_to_uint8(x):
    x = np.asarray(x, dtype=np.float32)
    xmax = x.max()
    if xmax <= 0:
        return np.zeros(x.shape, dtype=np.uint8)
    return np.clip(255 * (x / xmax), 0, 255).astype(np.uint8)

red = scale_to_uint8(merged)
green = scale_to_uint8(singleton)
blue = np.zeros_like(red, dtype=np.uint8)

rgb_video = np.stack([red, green, blue], axis=-1)   # (T, H, W, 3)

out_file = re.sub(r'\.tif$', '_caiman_events_singleton=green_merged=red.tif', video_filename)

tiff.imwrite(
    out_file,
    rgb_video,
    photometric="rgb"
)

In [ ]:
rows = []

for i, event in enumerate(merged_events):
    for linked_event in event:
        row = linked_event.copy()
        row["event_id"] = i
        rows.append(row)

events_df = pd.DataFrame(rows)
events_df

In [ ]:
os.chdir(event_dir)
events_df.to_csv(output_csv_filename, index=False)